# Script 2.98 - Combining all countries into one & re-assigning province names
11/07/2026, Kuba Kowalski 

This notebook first integrates the Cote d'Ivoire and Rwanda (2012) provinces into the file of other 21 countries. This script was previously part of 2.99_visualize-all, but I split it since 2.99 should be focused solely on data visualization instead of processing intermediate data. 

A new addition is the re-introduction of fully written-out province names instead of codes. 

In [1]:
# packages
import geopandas as gpd
import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
from pathlib import Path
from scipy.stats import gaussian_kde
import pandas as pd


## Alternative approach to integrating Cote d'Ivoire and Rwanda 2012 in combined countries file.
Rwanda's 2012 sample uses different admin boundaries than the world file. The geom column appears to be differently encoded and PostGIS will reject and attempts at union between the world and rwanda files due to an apparent column mismatch despite the same column name and type. Just like Cote, it must added using an alternative approach. 

Cote was created by manually aggregating lvl2 units into provinces from a separate shapefile than the world one provided by IPUMS. This process introduced an error in the ID column of the cote file that refuses to be fixed in postgis and prevents it from being appended to the file combining other countries.

Solution is to just manually realign only the relevant columns and brute force the combine. It works with Python, but not PostGIS.

In [2]:
# Birthplace: add Côte d'Ivoire and Rwanda 2012

main = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all.geojson"
)

cote = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\04cote\geom_edu_birthplace_4cote.geojson"
)

rwanda = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\12rwanda\geom_edu_birthplace_12rwanda2012.geojson"
)

# minimal harmonisation
cote = cote.copy()
cote["country_id"] = 4
cote["country"] = "cote_divoire"

rwanda = rwanda.copy()
rwanda["country_id"] = 12
rwanda["country"] = "rwanda"

rename_map = {
    "nc_cote": "n",
    "nc_rwanda": "n",
}

cote = cote.rename(columns={k: v for k, v in rename_map.items() if k in cote.columns})
rwanda = rwanda.rename(columns={k: v for k, v in rename_map.items() if k in rwanda.columns})

# match CRS
if cote.crs != main.crs:
    cote = cote.to_crs(main.crs)

if rwanda.crs != main.crs:
    rwanda = rwanda.to_crs(main.crs)

# Rwanda 2012 geometry uses PROV2012 instead of GEOLEVEL1
if "GEOLEVEL1" not in rwanda.columns and "PROV2012" in rwanda.columns:
    rwanda = rwanda.rename(columns={"PROV2012": "GEOLEVEL1"})

rwanda["GEOLEVEL1"] = rwanda["GEOLEVEL1"].astype(str).str.strip().str.zfill(6)

# keep only columns that already exist in the main file
common_cols = [c for c in main.columns if c in cote.columns and c in rwanda.columns]

combined = gpd.GeoDataFrame(
    pd.concat(
        [main[common_cols], cote[common_cols], rwanda[common_cols]],
        ignore_index=True
    ),
    geometry="geometry",
    crs=main.crs
)

combined.to_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson",
    driver="GeoJSON"
)

print(len(main), "rows in original birthplace")
print(len(cote), "rows in cote birthplace")
print(len(rwanda), "rows in rwanda birthplace")
print(len(combined), "rows in combined birthplace")
print(combined["country"].unique())

1947 rows in original birthplace
63 rows in cote birthplace
35 rows in rwanda birthplace
2045 rows in combined birthplace
['benin' 'botswana' 'burkina_faso' 'ghana' 'guinea' 'kenya' 'malawi'
 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo' 'uganda'
 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


In [3]:
# Residence: add Côte d'Ivoire and Rwanda 2012

main = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all.geojson"
)

cote = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\04cote\geom_edu_residence_4cote.geojson"
)

rwanda = gpd.read_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\12rwanda\geom_edu_residence_12rwanda2012.geojson"
)

cote = cote.copy()
cote["country_id"] = 4
cote["country"] = "cote_divoire"

rwanda = rwanda.copy()
rwanda["country_id"] = 12
rwanda["country"] = "rwanda"

rename_map = {
    "nc_cote": "n",
    "nc_rwanda": "n",
}

cote = cote.rename(columns={k: v for k, v in rename_map.items() if k in cote.columns})
rwanda = rwanda.rename(columns={k: v for k, v in rename_map.items() if k in rwanda.columns})

if cote.crs != main.crs:
    cote = cote.to_crs(main.crs)

if rwanda.crs != main.crs:
    rwanda = rwanda.to_crs(main.crs)

# Rwanda 2012 geometry uses PROV2012 instead of GEOLEVEL1
if "GEOLEVEL1" not in rwanda.columns and "PROV2012" in rwanda.columns:
    rwanda = rwanda.rename(columns={"PROV2012": "GEOLEVEL1"})

rwanda["GEOLEVEL1"] = rwanda["GEOLEVEL1"].astype(str).str.strip().str.zfill(6)    

common_cols = [c for c in main.columns if c in cote.columns and c in rwanda.columns]

combined = gpd.GeoDataFrame(
    pd.concat(
        [main[common_cols], cote[common_cols], rwanda[common_cols]],
        ignore_index=True
    ),
    geometry="geometry",
    crs=main.crs
)

combined.to_file(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson",
    driver="GeoJSON"
)

print(len(main), "rows in original residence")
print(len(cote), "rows in cote residence")
print(len(rwanda), "rows in rwanda residence")
print(len(combined), "rows in combined residence")
print(combined["country"].unique())

2027 rows in original residence
63 rows in cote residence
35 rows in rwanda residence
2125 rows in combined residence
['benin' 'botswana' 'burkina_faso' 'ethiopia' 'ghana' 'guinea' 'kenya'
 'malawi' 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo'
 'uganda' 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


## Check
If Cote and Rwanda not in output list, you are likely missing their GEOJSON files and should run all scripts starting with 1.XX. Afterwards, export their outputs as GEOJSONs through QGIS. 

Once the above is done, run this notebook again. 

In [4]:
input_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"

africa_outline_file = r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\1_inputs\IPUMS_boundaries\world_countries_africa_only.json"

output_dir = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa\3_outputs\99all\residence_higher_maps"
)
output_dir.mkdir(exist_ok=True)

gdf = gpd.read_file(input_file)
africa_outline = gpd.read_file(africa_outline_file)

print(len(gdf), "rows loaded from combined residence file")
print(gdf["country"].unique())

2125 rows loaded from combined residence file
['benin' 'botswana' 'burkina_faso' 'ethiopia' 'ghana' 'guinea' 'kenya'
 'malawi' 'mali' 'mozambique' 'senegal' 'sierra_leone' 'tanzania' 'togo'
 'uganda' 'zambia' 'liberia' 'cameroon' 'south_sudan' 'sudan' 'zimbabwe'
 'cote_divoire' 'rwanda']


## Re-adding province names for convenience 

For improved readability, the cells below add fully written province names to every object. 

In [16]:
# ------------------------------------------------------------------
# ADD PROVINCE NAMES FROM BOUNDARY FILES
# ------------------------------------------------------------------

world_boundary_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
    r"\1_inputs\IPUMS_boundaries\world_geolev1_2021\world_geolev1_2021.shp"
)

cote_boundary_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
    r"\1_inputs\4cotedivoire\geo1_ci1998.shp"
)

rwanda_boundary_file = Path(
    r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
    r"\1_inputs\12rwanda\geo1_rw2012\geo1_rw2012.shp"
)

world = gpd.read_file(world_boundary_file)
cote_names = gpd.read_file(cote_boundary_file)
rwanda_names = gpd.read_file(rwanda_boundary_file)

print("World columns:", world.columns.tolist())
print("Côte d'Ivoire columns:", cote_names.columns.tolist())
print("Rwanda columns:", rwanda_names.columns.tolist())

def clean_geolevel1(x):
    if pd.isna(x):
        return pd.NA

    x = str(x).strip()

    if x.endswith(".0"):
        x = x[:-2]

    return x.zfill(6)

# ------------------------------------------------------------------
# STANDARDIZE WORLD LOOKUP
# Replace ADMIN_NAME if the actual province-name column differs
# ------------------------------------------------------------------

world_lookup = world[
    ["GEOLEVEL1", "ADMIN_NAME"]
].copy()

world_lookup = world_lookup.rename(
    columns={"ADMIN_NAME": "province"}
)

world_lookup["GEOLEVEL1"] = (
    world_lookup["GEOLEVEL1"]
    .apply(clean_geolevel1)
)

# ------------------------------------------------------------------
# STANDARDIZE CÔTE D'IVOIRE LOOKUP
# ------------------------------------------------------------------
# Cote province code given in IPUM1998 column following a 3-number format. Convert to GEOLEVEL1 format by prefixing with "384" and ensuring 6-digit format.

cote_lookup = cote_names[
    ["IPUM1998", "ADMIN_NAME"]
].copy()

cote_lookup = cote_lookup.rename(
    columns={"ADMIN_NAME": "province"}
)

cote_lookup["GEOLEVEL1"] = (
    "384"
    + cote_lookup["IPUM1998"]
      .astype(str)
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.zfill(3)
)

cote_lookup = cote_lookup[
    ["GEOLEVEL1", "province"]
].copy()

# ------------------------------------------------------------------
# STANDARDIZE RWANDA 2012 LOOKUP
# ------------------------------------------------------------------
# Rwanda province code given in IPUM2012 column following a 3-number format. Convert to GEOLEVEL1 format by prefixing with "646" and ensuring 6-digit format.

rwanda_lookup = rwanda_names[
    ["IPUM2012", "ADMIN_NAME"]
].copy()

rwanda_lookup = rwanda_lookup.rename(
    columns={"ADMIN_NAME": "province"}
)

rwanda_lookup["GEOLEVEL1"] = (
    "000"
    + rwanda_lookup["IPUM2012"]
      .astype(str)
      .str.strip()
      .str.replace(r"\.0$", "", regex=True)
      .str.zfill(3)
)

rwanda_lookup = rwanda_lookup[
    ["GEOLEVEL1", "province"]
].copy()

# ------------------------------------------------------------------
# COMBINE LOOKUPS
# Rwanda and Côte d'Ivoire placed last so their dedicated files win
# ------------------------------------------------------------------

province_lookup = pd.concat(
    [
        world_lookup,
        cote_lookup,
        rwanda_lookup
    ],
    ignore_index=True
)

province_lookup = (
    province_lookup
    .dropna(subset=["GEOLEVEL1"])
    .drop_duplicates(
        subset=["GEOLEVEL1"],
        keep="last"
    )
)

# ------------------------------------------------------------------
# ADD PROVINCE TO GDF
# ------------------------------------------------------------------

gdf["GEOLEVEL1"] = (
    gdf["GEOLEVEL1"]
    .apply(clean_geolevel1)
)

if "province" in gdf.columns:
    gdf = gdf.drop(columns="province")

original_rows = len(gdf)

gdf = gdf.merge(
    province_lookup,
    on="GEOLEVEL1",
    how="left",
    validate="many_to_one"
)

if len(gdf) != original_rows:
    raise ValueError("Row count changed after adding province names.")

# Put province immediately after GEOLEVEL1
columns = list(gdf.columns)
columns.remove("province")
columns.insert(
    columns.index("GEOLEVEL1") + 1,
    "province"
)

gdf = gdf[columns]

# ------------------------------------------------------------------
# CHECK RESULTS
# ------------------------------------------------------------------

print(
    "Matched province names:",
    gdf["province"].notna().sum(),
    "of",
    len(gdf)
)

print("\nUnmatched GEOLEVEL1 values:")
print(
    gdf.loc[
        gdf["province"].isna(),
        ["GEOLEVEL1", "country"]
    ]
    .drop_duplicates()
    .sort_values(["country", "GEOLEVEL1"])
)

print("\nColumns:")
print(gdf.columns.tolist())

# ------------------------------------------------------------------
# ADD PROVINCE NAMES TO BOTH RESIDENCE AND BIRTHPLACE FILES
# ------------------------------------------------------------------
# Note: This overwrites both input files. 

files_to_update = [
    Path(
        r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
        r"\3_outputs\99all\geom_edu_residence_all_with_cote.geojson"
    ),
    Path(
        r"C:\Users\kowal010\Documents\rhi\research_assistant_rhi2026_africa"
        r"\3_outputs\99all\geom_edu_birthplace_all_with_cote.geojson"
    ),
]

for file_path in files_to_update:

    gdf_update = gpd.read_file(file_path)

    gdf_update["GEOLEVEL1"] = (
        gdf_update["GEOLEVEL1"]
        .apply(clean_geolevel1)
    )

    if "province" in gdf_update.columns:
        gdf_update = gdf_update.drop(columns="province")

    original_rows = len(gdf_update)

    gdf_update = gdf_update.merge(
        province_lookup,
        on="GEOLEVEL1",
        how="left",
        validate="many_to_one"
    )

    if len(gdf_update) != original_rows:
        raise ValueError(
            f"Row count changed while updating {file_path.name}"
        )

    columns = list(gdf_update.columns)
    columns.remove("province")
    columns.insert(
        columns.index("GEOLEVEL1") + 1,
        "province"
    )

    gdf_update = gdf_update[columns]

    gdf_update.to_file(
        file_path,
        driver="GeoJSON"
    )

    print(
        f"{file_path.name}: "
        f"{gdf_update['province'].notna().sum()} of "
        f"{len(gdf_update)} rows matched"
    )

World columns: ['CNTRY_NAME', 'ADMIN_NAME', 'CNTRY_CODE', 'GEOLEVEL1', 'BPL_CODE', 'geometry']
Côte d'Ivoire columns: ['CNTRY_NAME', 'ADMIN_NAME', 'CNTRY_CODE', 'IPUM1998', 'REGI1998', 'PARENT', 'geometry']
Rwanda columns: ['CNTRY_NAME', 'ADMIN_NAME', 'CNTRY_CODE', 'IPUM2012', 'PROV2012', 'PARENT', 'geometry']
Matched province names: 2125 of 2125

Unmatched GEOLEVEL1 values:
Empty DataFrame
Columns: [GEOLEVEL1, country]
Index: []

Columns:
['country_id', 'country', 'GEOLEVEL1', 'province', 'cohort', 'primary_educ', 'higher_educ', 'tertiary_educ', 'n', 'geometry']
geom_edu_residence_all_with_cote.geojson: 2125 of 2125 rows matched
geom_edu_birthplace_all_with_cote.geojson: 2045 of 2045 rows matched
